## 1. Environment Setup

In [ ]:
# Install required packages
# IMPORTANT: After this cell finishes, go to Runtime -> Restart session,
# then run all cells again from the top.
!pip install -q transformers==4.44.0 accelerate==0.33.0 datasets scipy scikit-learn

import torch
import numpy as np
print(f'PyTorch : {torch.__version__}')
print(f'NumPy   : {np.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
print()
print('Packages ready.  Restart the runtime now (Runtime -> Restart session),')
print('then run all cells again from the top.')

In [ ]:
MOUNT_DRIVE = True   # set False if uploading files directly

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/AuditQuant'
    import os; os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f'Drive mounted at {DRIVE_DIR}')

## 2. Upload / Locate Dataset

In [ ]:
import os, json
from pathlib import Path

# Option A: upload from local machine
# from google.colab import files
# uploaded = files.upload()   # select dataset_v2.json
# DATASET_PATH = Path(list(uploaded.keys())[0])

# Option B: copy from Drive (adjust path if needed)
DATASET_PATH = Path('/content/drive/MyDrive/AuditQuant/dataset_v2.json')

# Option C: already uploaded to /content/
# DATASET_PATH = Path('/content/dataset_v2.json')

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f'{DATASET_PATH} not found.
'
        'Generate it locally with:
'
        '  python evaluation/scripts/prepare_dataset_v2.py
'
        'Then upload dataset_v2.json to Google Drive at MyDrive/AuditQuant/dataset_v2.json'
    )

with DATASET_PATH.open() as f:
    DATASET = json.load(f)

VULN_LABELS = DATASET['vuln_labels']
N_LABELS    = DATASET['n_labels']
print(f'Dataset version : {DATASET.get("version", "unknown")}')
print(f'Total samples   : {DATASET["n_total"]} '
      f'(train={DATASET["n_train"]} val={DATASET["n_val"]} test={DATASET["n_test"]})')
print(f'Label classes ({N_LABELS}): {VULN_LABELS}')
print()
print('Label counts across full dataset:')
for label, cnt in DATASET.get('label_counts', {}).items():
    bar = '#' * min(cnt // 50, 40)
    print(f'  {label:<30} {cnt:>5}  {bar}')
print()
print('Sources:')
for src, cnt in DATASET.get('sources', {}).items():
    print(f'  {src:<30} {cnt}')


## 3. Training Configuration

In [ ]:
CONFIG = {
    'model_name':          'microsoft/codebert-base',
    'max_seq_len':         512,
    'batch_size':          16,
    'grad_accum_steps':    2,       # effective batch = 32
    'learning_rate':       2e-5,
    'weight_decay':        0.01,
    'epochs':              20,
    'warmup_ratio':        0.05,
    'freeze_layers':       6,       # unfreeze top 6 of 12 transformer layers
    'dropout':             0.15,
    'risk_head':           True,
    'risk_loss_weight':    0.1,
    'focal_gamma':         2.0,     # focal loss gamma
    'label_smoothing':     0.0,
    'early_stop_patience': 5,
    'threshold':           0.5,     # optimised per-class on val split
    'fp16':                True,
    'seed':                42,
}

if not torch.cuda.is_available():
    CONFIG['fp16'] = False
    CONFIG['batch_size'] = 4
    print('No GPU - fp16 disabled, batch_size -> 4')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device          : {DEVICE}')
print(f'Effective batch : {CONFIG["batch_size"] * CONFIG["grad_accum_steps"]}')
print(f'Unfrozen layers : {12 - CONFIG["freeze_layers"]} / 12')


## 4. Dataset Preparation

In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

torch.manual_seed(CONFIG['seed'])
rng = np.random.default_rng(CONFIG['seed'])

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])


class ContractDataset(Dataset):
    def __init__(self, samples, max_len=512):
        self.samples = samples
        self.max_len  = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        enc = tokenizer(
            s['source_code'],
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(s['label_vector'],  dtype=torch.float32),
            'risk':           torch.tensor([s['risk_score']], dtype=torch.float32),
        }


train_samples = [s for s in DATASET['samples'] if s['split'] == 'train']
val_samples   = [s for s in DATASET['samples'] if s['split'] == 'val']
test_samples  = [s for s in DATASET['samples'] if s['split'] == 'test']

train_ds = ContractDataset(train_samples, CONFIG['max_seq_len'])
val_ds   = ContractDataset(val_samples,   CONFIG['max_seq_len'])
test_ds  = ContractDataset(test_samples,  CONFIG['max_seq_len'])

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

label_matrix      = np.array([s['label_vector'] for s in train_samples])
true_pos_counts   = label_matrix.sum(axis=0)          # actual, unclipped
active_label_mask = (true_pos_counts > 0)             # bool, shape (n_labels,)
active_labels     = [VULN_LABELS[i] for i in range(N_LABELS) if  active_label_mask[i]]
zero_labels       = [VULN_LABELS[i] for i in range(N_LABELS) if not active_label_mask[i]]
active_idx        = np.where(active_label_mask)[0]    # index array for sklearn

if zero_labels:
    print(f'Warning: {len(zero_labels)} label(s) have 0 positive training examples.')
    print(f'  Excluded from loss + metrics: {zero_labels}')
    print(f'  (To include them, merge this dataset with the original manifest-based dataset.)')

# Inverse-frequency weighting (neg/pos), clipped to [1, 20].
# Zero-support labels get pos_weight=1.0 - irrelevant since they are masked.
pos_counts   = true_pos_counts.clip(min=1)
neg_counts   = len(train_samples) - pos_counts
raw_weights  = np.where(active_label_mask, neg_counts / pos_counts, 1.0)
class_weights = torch.tensor(np.clip(raw_weights, 1.0, 20.0), dtype=torch.float32)
active_label_mask_t = torch.tensor(active_label_mask, dtype=torch.float32)

print(f'\nTrain: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')
print(f'\nLabel distribution in training split:')
for label, cnt, w in zip(VULN_LABELS, true_pos_counts.astype(int), class_weights.numpy()):
    status = '' if cnt > 0 else '  [INACTIVE]'
    bar = '#' * min(int(cnt // 20), 40)   # 1 char per 20 contracts, max 40
    print(f'  {label:<30} n={cnt:4d}  pos_weight={w:5.1f}  {bar}{status}')

## 5. Model Definition

In [ ]:
import torch.nn as nn
from transformers import AutoModel


class CodeBERTVulnModel(nn.Module):
    def __init__(self, n_labels, risk_head=True, model_name='microsoft/codebert-base',
                 dropout=0.1, freeze_layers=6):
        super().__init__()
        self.n_labels   = n_labels
        self.has_risk   = risk_head
        self.model_name = model_name

        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size  # 768

        for i, layer in enumerate(self.backbone.encoder.layer):
            if i < freeze_layers:
                for p in layer.parameters():
                    p.requires_grad = False

        self.dropout   = nn.Dropout(dropout)
        self.vuln_head = nn.Linear(hidden, n_labels)

        if risk_head:
            self.risk_head = nn.Sequential(
                nn.Linear(hidden, 64),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(64, 1),
                nn.Sigmoid(),
            )

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(out.last_hidden_state[:, 0, :])
        result = {'logits': self.vuln_head(cls)}
        if self.has_risk:
            result['risk'] = self.risk_head(cls)
        return result


model = CodeBERTVulnModel(
    n_labels=N_LABELS,
    risk_head=CONFIG['risk_head'],
    model_name=CONFIG['model_name'],
    dropout=CONFIG['dropout'],
    freeze_layers=CONFIG['freeze_layers'],
).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total:,} total, {trainable:,} trainable ({100*trainable/total:.1f}%)')

## 6. Training Loop

In [ ]:
import math
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, precision_score, recall_score


class FocalBCELoss(torch.nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None, label_mask=None):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight          # (n_labels,) tensor
        # label_mask: (n_labels,) float32 - 1.0 active, 0.0 skip
        if label_mask is not None:
            self.register_buffer('label_mask', label_mask)
        else:
            self.label_mask = None

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets,
            pos_weight=self.pos_weight,
            reduction='none',
        )
        p   = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        focal = (1 - p_t) ** self.gamma * bce
        if self.label_mask is not None:
            focal = focal * self.label_mask
        return focal.mean()


def compute_metrics(all_logits, all_labels, threshold=0.5, active_idx=None):
    """Compute F1/P/R. If active_idx is given, only score those columns."""
    from scipy.special import expit
    probs = expit(all_logits)
    preds = (probs >= threshold).astype(int)
    if active_idx is not None:
        all_labels = all_labels[:, active_idx]
        preds      = preds[:, active_idx]
    macro_f1 = f1_score(all_labels, preds, average='macro', zero_division=0)
    micro_f1 = f1_score(all_labels, preds, average='micro', zero_division=0)
    prec     = precision_score(all_labels, preds, average='macro', zero_division=0)
    rec      = recall_score(all_labels, preds, average='macro', zero_division=0)
    return {'macro_f1': macro_f1, 'micro_f1': micro_f1, 'precision': prec, 'recall': rec}


def optimise_thresholds(val_logits, val_labels, n_thresholds=17, active_mask=None):
    """Find per-class threshold that maximises F1 (active classes only)."""
    from scipy.special import expit
    probs      = expit(val_logits)
    thresholds = np.linspace(0.1, 0.9, n_thresholds)
    best_t     = np.full(val_labels.shape[1], 0.5)
    for c in range(val_labels.shape[1]):
        if active_mask is not None and not active_mask[c]:
            continue   # leave at 0.5; this label is never predicted
        best_f1 = 0.0
        for t in thresholds:
            preds_c = (probs[:, c] >= t).astype(int)
            f1 = f1_score(val_labels[:, c], preds_c, zero_division=0)
            if f1 > best_f1:
                best_f1   = f1
                best_t[c] = t
    return best_t   # shape (n_labels,)


optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay'],
)

total_steps  = math.ceil(len(train_loader) / CONFIG['grad_accum_steps']) * CONFIG['epochs']
warmup_steps = int(total_steps * CONFIG['warmup_ratio'])
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

scaler    = GradScaler(enabled=CONFIG['fp16'])
criterion = FocalBCELoss(
    gamma=CONFIG['focal_gamma'],
    pos_weight=class_weights.to(DEVICE),
    label_mask=active_label_mask_t.to(DEVICE),
)
mse_loss  = torch.nn.MSELoss()

history       = []
best_val_f1   = 0.0
patience_left = CONFIG['early_stop_patience']
best_ckpt     = {}
best_thresholds = np.full(N_LABELS, 0.5)

print(f'Total optimiser steps : {total_steps}')
print(f'Warmup steps          : {warmup_steps}')
print(f'Active labels scored  : {len(active_labels)} / {N_LABELS}')
print()

for epoch in range(1, CONFIG['epochs'] + 1):
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)
        risk_targets   = batch['risk'].to(DEVICE)

        with autocast(enabled=CONFIG['fp16']):
            out  = model(input_ids, attention_mask)
            loss = criterion(out['logits'], labels)
            if CONFIG['risk_head'] and 'risk' in out:
                loss = loss + CONFIG['risk_loss_weight'] * mse_loss(out['risk'], risk_targets)
            loss = loss / CONFIG['grad_accum_steps']

        scaler.scale(loss).backward()
        train_loss += loss.item() * CONFIG['grad_accum_steps']

        if (step + 1) % CONFIG['grad_accum_steps'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_logits_all, val_labels_all, val_risk_all = [], [], []

    with torch.no_grad():
        for batch in val_loader:
            out = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
            val_logits_all.append(out['logits'].cpu().numpy())
            val_labels_all.append(batch['labels'].numpy())
            if 'risk' in out:
                val_risk_all.append(out['risk'].squeeze(-1).cpu().numpy())

    val_logits  = np.concatenate(val_logits_all)
    val_labels  = np.concatenate(val_labels_all)
    # Score only active labels so inactive classes don't dilute macro-F1
    val_metrics = compute_metrics(val_logits, val_labels, CONFIG['threshold'],
                                  active_idx=active_idx)

    row = {
        'epoch':         epoch,
        'train_loss':    round(avg_train_loss, 4),
        'val_macro_f1':  round(val_metrics['macro_f1'], 4),
        'val_micro_f1':  round(val_metrics['micro_f1'], 4),
        'val_precision': round(val_metrics['precision'], 4),
        'val_recall':    round(val_metrics['recall'], 4),
    }
    history.append(row)

    improved = val_metrics['macro_f1'] > best_val_f1
    marker   = ' *' if improved else ''
    print(f'Epoch {epoch:2d}/{CONFIG["epochs"]} | loss={avg_train_loss:.4f} | '
          f'val F1={val_metrics["macro_f1"]:.4f}  '
          f'P={val_metrics["precision"]:.3f}  R={val_metrics["recall"]:.3f}{marker}')

    if improved:
        best_val_f1     = val_metrics['macro_f1']
        patience_left   = CONFIG['early_stop_patience']
        best_thresholds = optimise_thresholds(val_logits, val_labels,
                                              active_mask=active_label_mask)
        best_ckpt = {
            'epoch':                epoch,
            'model_state_dict':     {k: v.cpu() for k, v in model.state_dict().items()},
            'model_name':           CONFIG['model_name'],
            'n_labels':             N_LABELS,
            'vuln_labels':          VULN_LABELS,
            'active_labels':        active_labels,
            'risk_head':            CONFIG['risk_head'],
            'val_macro_f1':         best_val_f1,
            'per_class_thresholds': best_thresholds.tolist(),
            'config':               CONFIG,
        }
    else:
        patience_left -= 1
        if patience_left == 0:
            print(f'\nEarly stopping at epoch {epoch} (patience exhausted)')
            break

print(f'\nTraining complete.  Best val macro-F1 (active labels): {best_val_f1:.4f}')
print(f'Per-class thresholds: {[round(t, 2) for t in best_thresholds.tolist()]}')

## 7. Test-Set Evaluation

In [ ]:
from scipy.special import expit
from sklearn.metrics import classification_report, precision_recall_fscore_support

# Restore best checkpoint
model.load_state_dict(best_ckpt['model_state_dict'])
model.to(DEVICE).eval()

test_logits_all, test_labels_all, test_risk_all = [], [], []
with torch.no_grad():
    for batch in test_loader:
        out = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
        test_logits_all.append(out['logits'].cpu().numpy())
        test_labels_all.append(batch['labels'].numpy())
        if 'risk' in out:
            test_risk_all.append(out['risk'].squeeze(-1).cpu().numpy())

test_logits = np.concatenate(test_logits_all)
test_labels = np.concatenate(test_labels_all)
test_probs  = expit(test_logits)

per_class_t = np.array(best_ckpt['per_class_thresholds'])
test_pred   = (test_probs >= per_class_t).astype(int)

print('Per-class decision thresholds (optimised on val split):')
for label, t, active in zip(VULN_LABELS, per_class_t, active_label_mask):
    status = '' if active else '  [inactive]'
    print(f'  {label:<30} {t:.2f}{status}')

# Classification report on active labels only
print('\n--- Test Set Classification Report (active labels) ---')
print(classification_report(
    test_labels[:, active_idx],
    test_pred[:, active_idx],
    target_names=active_labels,
    zero_division=0,
))

test_metrics_opt = {
    'macro_f1':  float(f1_score(test_labels[:, active_idx], test_pred[:, active_idx],
                                average='macro',  zero_division=0)),
    'micro_f1':  float(f1_score(test_labels[:, active_idx], test_pred[:, active_idx],
                                average='micro',  zero_division=0)),
    'precision': float(precision_score(test_labels[:, active_idx], test_pred[:, active_idx],
                                       average='macro', zero_division=0)),
    'recall':    float(recall_score(test_labels[:, active_idx], test_pred[:, active_idx],
                                    average='macro', zero_division=0)),
}
print(f'Test macro-F1  (per-class threshold, active labels): {test_metrics_opt["macro_f1"]:.4f}')
print(f'Test micro-F1  (per-class threshold, active labels): {test_metrics_opt["micro_f1"]:.4f}')
print(f'Test precision (per-class threshold, active labels): {test_metrics_opt["precision"]:.4f}')
print(f'Test recall    (per-class threshold, active labels): {test_metrics_opt["recall"]:.4f}')

risk_mae  = None
risk_corr = None
if test_risk_all:
    test_risk = np.concatenate(test_risk_all)
    true_risk = np.array([s['risk_score'] for s in test_samples])
    risk_mae  = float(np.mean(np.abs(test_risk - true_risk)))
    risk_corr = float(np.corrcoef(test_risk, true_risk)[0, 1]) if len(true_risk) > 2 else 0.0
    print(f'Risk MAE      : {risk_mae:.4f}')
    print(f'Risk Pearson r: {risk_corr:.4f}')

## 8. Save Artifacts

In [ ]:
import datetime

# Save checkpoint
ckpt_path = Path('/content/checkpoint_best.pt')
torch.save(best_ckpt, str(ckpt_path))
print(f'Checkpoint saved: {ckpt_path} ({ckpt_path.stat().st_size/1024:.0f} KB)')

# Save training history
history_path = Path('/content/training_history.json')
with history_path.open('w') as f:
    json.dump({'config': CONFIG, 'history': history}, f, indent=2)
print(f'Training history: {history_path}')

# Save eval results
p_per, r_per, f_per, sup = precision_recall_fscore_support(
    test_labels, test_pred, zero_division=0
)
per_class = {
    label: {
        'precision': round(float(p_per[i]), 4),
        'recall':    round(float(r_per[i]), 4),
        'f1':        round(float(f_per[i]), 4),
        'support':   int(sup[i]),
        'threshold': round(float(per_class_t[i]), 4),
        'active':    bool(active_label_mask[i]),
    }
    for i, label in enumerate(VULN_LABELS)
}

per_sample = []
for i, s in enumerate(test_samples):
    per_sample.append({
        'contract_id':     s['contract_id'],
        'true_labels':     s['vuln_types'],
        'pred_labels':     [VULN_LABELS[j] for j in range(N_LABELS) if test_pred[i, j]],
        'probs':           [round(float(p), 4) for p in test_probs[i]],
        'pred_risk_score': round(float(test_risk[i]), 4) if test_risk_all else 0.0,
        'true_risk_score': s['risk_score'],
        'defi_category':   s.get('defi_category', 'other'),
    })

eval_results = {
    'split':                'test',
    'per_class_thresholds': per_class_t.tolist(),
    'active_labels':        active_labels,
    'zero_support_labels':  zero_labels,
    'n_samples':            len(test_samples),
    'aggregate': {
        'macro_precision': round(test_metrics_opt['precision'], 4),
        'macro_recall':    round(test_metrics_opt['recall'], 4),
        'macro_f1':        round(test_metrics_opt['macro_f1'], 4),
        'micro_f1':        round(test_metrics_opt['micro_f1'], 4),
        'note':            'Computed over active labels only (zero-support labels excluded)',
    },
    'per_class':            per_class,
    'risk_metrics':         {'mae': round(risk_mae, 4) if risk_mae is not None else None,
                             'pearson_r': round(risk_corr, 4) if risk_corr is not None else None},
    'per_sample':           per_sample,
    'training_history':     history,
    'vuln_labels':          VULN_LABELS,
    'checkpoint_config':    CONFIG,
    'created_at':           datetime.datetime.utcnow().isoformat() + 'Z',
}
eval_path = Path('/content/eval_results.json')
with eval_path.open('w') as f:
    json.dump(eval_results, f, indent=2)
print(f'Eval results  : {eval_path}')

# Copy to Drive if mounted
if MOUNT_DRIVE:
    import shutil
    for src in [ckpt_path, history_path, eval_path]:
        dst = Path(DRIVE_DIR) / src.name
        shutil.copy(str(src), str(dst))
        print(f'Copied to Drive: {dst}')

## 9. Download Artifacts

In [ ]:
# Download to your local machine
from google.colab import files
files.download('/content/checkpoint_best.pt')
files.download('/content/eval_results.json')
files.download('/content/training_history.json')

print('After downloading, place checkpoint_best.pt at:')
print('  evaluation/llm_training/checkpoints/checkpoint_best.pt')
print('and eval_results.json at:')
print('  evaluation/results/llm_eval.json')
print()
print('Then run locally:')
print('  python evaluation/scripts/compare_llm_vs_tools.py')

## 10. Training Curves (Optional)

In [ ]:
import matplotlib.pyplot as plt

epochs_done  = [h['epoch']         for h in history]
train_losses = [h['train_loss']    for h in history]
val_f1s      = [h['val_macro_f1']  for h in history]
val_precs    = [h['val_precision'] for h in history]
val_recs     = [h['val_recall']    for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_done, train_losses, 'b-o', markersize=4, label='Train Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training Loss'); ax1.grid(alpha=0.4); ax1.legend()

ax2.plot(epochs_done, val_f1s,   'g-o', markersize=4, label='Val Macro-F1 (active)')
ax2.plot(epochs_done, val_precs, 'b--', markersize=3, label='Val Precision')
ax2.plot(epochs_done, val_recs,  'r--', markersize=3, label='Val Recall')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Score')
ax2.set_title('Validation Metrics (active labels only)'); ax2.grid(alpha=0.4); ax2.legend()

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: /content/training_curves.png')